In [2]:
!pip -q install -U datasets pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 110.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [3]:

import hashlib
import re
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

# ==================================================
# 1. Configuration
# ==================================================

DATASET_ID = "lmarena-ai/arena-human-preference-140k"
OUTPUT_DIR = Path("/content/arena_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = {
    "GPT": 0,
    "Claude": 1,
    "Gemini": 2
}

COLUMNS = [
    "prompt_id",
    "LLM_Input",
    "LLM_output",
    "LLM_name",
    "label"
]

SEED = 42

# ==================================================
# 2. Model family mapping
# ==================================================

def get_family(model_name):
    name = str(model_name or "").lower()

    if name.startswith(("gpt-", "chatgpt-")):
        return "GPT"

    if name.startswith("claude-"):
        return "Claude"

    if name.startswith("gemini-"):
        return "Gemini"

    return None


# ==================================================
# 3. Extract text
# ==================================================

def extract_text(message):
    if not isinstance(message, dict):
        return None

    content = message.get("content")

    if isinstance(content, str):
        return content.strip() or None

    if not isinstance(content, list) or not content:
        return None

    parts = []

    for block in content:
        if not isinstance(block, dict):
            return None

        if block.get("type") != "text":
            return None

        if (
            block.get("image") is not None
            or block.get("mimeType") is not None
        ):
            return None

        value = block.get("text")

        if not isinstance(value, str):
            return None

        parts.append(value)

    return "\n".join(parts).strip() or None


def extract_conversation(conversation):
    # Use text-only, single-turn conversations.
    if not isinstance(conversation, list):
        return None

    if len(conversation) != 2:
        return None

    user, assistant = conversation

    if not isinstance(user, dict):
        return None

    if not isinstance(assistant, dict):
        return None

    if user.get("role") != "user":
        return None

    if assistant.get("role") != "assistant":
        return None

    prompt = extract_text(user)
    response = extract_text(assistant)

    if not prompt or not response:
        return None

    return prompt, response


def get_prompt_id(prompt):
    normalized = re.sub(
        r"\s+", " ", prompt
    ).strip().casefold()

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


# ==================================================
# 4. Download the ENTIRE dataset
# ==================================================

print("Loading Arena dataset...")

dataset = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True
)

records = []
scanned = 0

for row in dataset:
    scanned += 1

    # English only
    if row.get("language") != "en":
        continue

    conv_a = extract_conversation(
        row.get("conversation_a")
    )

    conv_b = extract_conversation(
        row.get("conversation_b")
    )

    if conv_a is None or conv_b is None:
        continue

    prompt_a, response_a = conv_a
    prompt_b, response_b = conv_b

    # Both models must have received the same prompt.
    if get_prompt_id(prompt_a) != get_prompt_id(prompt_b):
        continue

    prompt_id = get_prompt_id(prompt_a)

    for model_name, response in [
        (row.get("model_a"), response_a),
        (row.get("model_b"), response_b)
    ]:
        family = get_family(model_name)

        if family is None:
            continue

        records.append({
            "prompt_id": prompt_id,
            "LLM_Input": prompt_a,
            "LLM_output": response,
            "LLM_name": family,
            "label": LABELS[family]
        })

    if scanned % 10000 == 0:
        print(f"Scanned {scanned:,} battles")

print(f"Total battles scanned: {scanned:,}")
print(f"Target responses extracted: {len(records):,}")


# ==================================================
# 5. Create DataFrame and remove duplicates
# ==================================================

df = pd.DataFrame.from_records(
    records,
    columns=COLUMNS
)

if df.empty:
    raise ValueError("No matching responses were extracted.")

df = df.drop_duplicates(
    subset=[
        "prompt_id",
        "LLM_name",
        "LLM_output"
    ]
).reset_index(drop=True)

print("\nModel distribution before balancing:")
print(df["LLM_name"].value_counts())


# ==================================================
# 6. Split by prompt to prevent data leakage
# ==================================================

split1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=SEED
)

train_idx, rest_idx = next(
    split1.split(df, groups=df["prompt_id"])
)

train = df.iloc[train_idx].copy()
rest = df.iloc[rest_idx].copy()

split2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(
    split2.split(
        rest,
        groups=rest["prompt_id"]
    )
)

splits = {
    "train": train,
    "validation": rest.iloc[val_idx].copy(),
    "test": rest.iloc[test_idx].copy()
}


# ==================================================
# 7. Balance classes independently in each split
# ==================================================

for name, part in splits.items():
    counts = part["LLM_name"].value_counts().reindex(
        LABELS.keys(),
        fill_value=0
    )

    n = int(counts.min())

    if n == 0:
        raise ValueError(
            f"{name} is missing a model family."
        )

    balanced = (
        part.groupby("LLM_name", group_keys=False)
        .sample(n=n, random_state=SEED)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )

    splits[name] = balanced[COLUMNS]


# ==================================================
# 8. Validate splits
# ==================================================

train_ids = set(splits["train"]["prompt_id"])
val_ids = set(splits["validation"]["prompt_id"])
test_ids = set(splits["test"]["prompt_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\nNo prompt leakage between splits.")


# ==================================================
# 9. Save CSV files
# ==================================================

for name, part in splits.items():
    path = OUTPUT_DIR / f"{name}.csv"

    part.to_csv(
        path,
        index=False,
        encoding="utf-8"
    )

    print(f"\n{name}: {len(part):,} rows")
    print(part["LLM_name"].value_counts())


combined = pd.concat(
    splits.values(),
    ignore_index=True
)

combined.to_csv(
    OUTPUT_DIR / "arena_processed_all.csv",
    index=False,
    encoding="utf-8"
)

print("\nFinal dataset:", combined.shape)
print("Columns:", combined.columns.tolist())
print("Saved to:", OUTPUT_DIR)


Loading Arena dataset...


README.md:   0%|          | 0.00/5.54k [00:00<?, ?B/s]

Scanned 20,000 battles
Scanned 40,000 battles
Scanned 60,000 battles
Scanned 90,000 battles
Scanned 100,000 battles
Total battles scanned: 135,634
Target responses extracted: 53,767

Model distribution before balancing:
LLM_name
Claude    25366
Gemini    18317
GPT        9999
Name: count, dtype: int64

No prompt leakage between splits.

train: 20,844 rows
LLM_name
GPT       6948
Gemini    6948
Claude    6948
Name: count, dtype: int64

validation: 4,779 rows
LLM_name
Claude    1593
GPT       1593
Gemini    1593
Name: count, dtype: int64

test: 4,374 rows
LLM_name
GPT       1458
Claude    1458
Gemini    1458
Name: count, dtype: int64

Final dataset: (29997, 5)
Columns: ['prompt_id', 'LLM_Input', 'LLM_output', 'LLM_name', 'label']
Saved to: /content/arena_processed
